## **Projeto de Aprendizagem Computacional 2**

### Grupo 1: Carlos Pereira, João Monteiro Dias, Maria Santos: 202404276 (pedir ups)

**Objetivo**: treinar um modelo capaz de detectar sopros cardíacos a partir de fonocardiogramas.

**Modelos escolhidos**: CNN2D e RNN 


In [3]:
#Imports
import source.config as config
from source.config import *
keras.utils.set_random_seed(42)
import source.runmodel as runmodel
import source.patientaggregator as patientaggregator
from source.helpers import md_table_build, weighted_accuracy

CLASSES = {
    0: "Present",
    1: "Unknown",
    2: "Absent"
}

METRIC_W=config.METRIC_W
CFG=config.CFG


## Melhor configuração de agrupamento de previsões 

Escolha de configuração baseada no WA:

- RNN: topk,5,unweighted,0.76115483045578

- CNN: topk,5,unweighted,0.780183732509613

In [4]:
#md_table_build(csv_file=PARAMS_PATH)

### **CNN2D** : Arquitetura

In [5]:
class MyModel(keras.Model):
    def __init__(self, name:str, **kwargs):
        super().__init__(name=name, **kwargs)


In [6]:
Custom_CNN2D:MyModel= models.Sequential(
    [
        layers.Input(shape=(153, 40, 1)),

        # Bloco 1: 16 filtros
        layers.Conv2D(16, 3, padding="same", activation="relu"),
        layers.BatchNormalization(momentum=0.9),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Bloco 2: 32 filtros
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(momentum=0.9),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Bloco 3: 64 filtros
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(momentum=0.9),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(len([key for key in CLASSES.values()]), activation="softmax"),
    ],
    name="Custom_CNN2D",
)

## RNN: arquitetura

In [7]:
McDonald_model:MyModel=models.Sequential(
    [
        layers.Input(shape=(CFG.time_steps,CFG.n_features)),
        layers.Bidirectional(layers.GRU(units=64, return_sequences=True)), #return_sequences=True; as camadas anteriores são passadas para layers seguintes. Essas camadas são passadas como sequências.
        layers.Bidirectional(layers.GRU(units=64, return_sequences=True)),
        layers.Bidirectional(layers.GRU(units=64)), #O output final é ATIVAÇÃO (computado a partir de pesos e inputs).
        layers.Dropout(0.1),
        #Bloco 2: : two-layer NN, fully-connected, c/ funções de ativação Tanh, c/tamanho 60 e 40
        layers.Dense(60, activation='tanh'),
        layers.Dropout(0.1),
        layers.Dense(40, activation='tanh'),
        #OUTPUT
        layers.Dense(len([key for key in CLASSES.values()]), activation='softmax') # 5 segmentation labels per time step
    ],  
    name="McDonald"
)

## Pré Processamento 
- extração das features .... joão
- split por paciente 
- normalização por janela...

## Loading data 

In [8]:
def load_split(architecture, name, CACHE:str|Path):
    cache = np.load(Path(CACHE) / f"{name}.npz")
    info = pd.read_csv(Path(CACHE) / f"{name}_info.csv", dtype={"pid": str})
    #Extrai do zip .npz o numpy array 
    return cache[f"{architecture}"].astype("float32"), cache["y"].astype(int), info["pid"].to_numpy(dtype=object)
    #Não verificamos pid porque todas as entradas têm o id do paciente.

## Estratégia de Treino

Foi desenhada para resolver os dois maiores desafios deste problema (a deteção de sopros cardíacos da competição CirCor DigiScope): o desbalanceamento severo das classes e o desfasamento entre a unidade de entrada do modelo (janela de áudio) e a unidade de diagnóstico clínico (paciente).

- A CNN 2D aprende características locais (tempo-frequência) analisando cada janela de tempo isolada.  

- Agregação por Paciente (to_patient): agrupa todas as janelas que partilham o mesmo pid e calcula a média das probabilidades previstas para esse doente:

    $$/bar{P}(/text{classe} /mid /text{paciente}) = /frac{1}{K} /sum_{k=1}^K P(/text{classe} /mid /text{janela}_k)$$

- Compensação de Classes Desbalanceadas no Treino (class_weight):   Absent: $/approx 73.8/%$   Present: $/approx 19.0/%$   Unknown: $/approx 7.2/%$. 
Para contrariar o desbalancemanto, calculamos os pesos de classe inversamente proporcionais às contagens do conjunto de treino:
    $$/text{weight}_i = /frac{N_{/text{total}}}{3 /times N_i}$$

- **Otimização Orientada à Métrica da Competição (Weighted Accuracy - WA)** : A métrica oficial da competição PhysioNet atribui recompensas desiguais aos acertos com base no vetor de pesos METRIC_W = [5.0, 3.0, 1.0]:  
 -  Present: peso $5.0$ |  Unknown: peso $3.0$ |  Absent: peso $1.0$

Esta otimização é implementada pelas:

- Regra de Decisão do Doente: mean, ou seja as as probabilidades médias eram ponderadas por este vetor (score = pr * METRIC_W) antes de passarem pelo argmax;

- Callback Customizado (PatientWA): no final de cada epoch, a função on_epoch_end corre a validação completa, agrega por paciente e calcula o WA real da validação (val_wa), permitindo monitorizar a weighted accuracy e não apenas a accuracy por janela;

**Gestão Dinâmica da Convergência (Callbacks)**: O treino utiliza dois mecanismos automáticos para evitar overfitting e garantir que a rede converge de forma estável:   
- ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3): Se o erro de validação (val_loss) não diminuir durante 3 épocas consecutivas, a taxa de aprendizagem (learning rate) é reduzida para metade (passando de $0.001$ para $0.0005$, depois $0.00025$, etc.), permitindo que a descida do gradiente refine os pesos.   
- EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True): vigia diretamente o val_wa ao nível do paciente. Se o modelo passar 8 épocas sem conseguir superar o seu recorde de WA, o treino é interrompido e os pesos da melhor época são automaticamente restaurados para o ficheiro final.

**Estas classes estão definidas no ficheiro 'train_strategy.py'**

------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
from source.kfold import split_fold
from source.helpers import export_csv

MODEL_CONFIG: dict[MyModel, str]={
    Custom_CNN2D:"cnn2d",
    McDonald_model:"rnn"
}


for model, architecture in MODEL_CONFIG.items():

    Xtr, ytr, ptr = load_split(architecture=architecture, name="train", CACHE=Path("C:/Users/user/Desktop/Code_Projects/AC_II_project/final_branch/AC_project/data"))
    Xva, yva, pva = load_split(architecture=architecture, name="val", CACHE=Path("C:/Users/user/Desktop/Code_Projects/AC_II_project/final_branch/AC_project/data"))

    X:np.ndarray=np.concatenate([Xtr, Xva])
    y:np.ndarray=np.concatenate([ytr, yva])
    pid:np.ndarray=np.concatenate([ptr, pva])

    for fold in range(CFG.K):
        fit_idx, in_idx, test_idx = split_fold(y=y, pid=pid, k=config.CFG.K, fold=fold)
        DATA=dict(
            X=X, 
            y=y, 
            pid=pid,
            fit_idx=fit_idx,
            in_idx=in_idx,
            test_idx=test_idx
        )
        keras.utils.set_random_seed(config.CFG.seed)
        model=keras.models.clone_model(Custom_CNN2D)
        runmodel.run_model(
            model=model,    
            weight_mode="smoothed",
            aggregator=patientaggregator.agg,
            data=DATA,
            epochs_requested=CFG.epochs,
            seed=CFG.seed,
            fold=fold
        )


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\user\\Desktop\\Code_Projects\\AC_II_project\\final_branch\\AC_project\\data\\train.npz\\train.npz'

## CNN2D: Resultados com a estratégia inicial



- Treino com 1 split: 

        A_murmur: 0.6778

                              |True Present | True Unknown | True Absent
                Pred Present  |          27 |          4   |       62
                Pred Unknown  |          0  |          6   |       13
                Pred Absent   |          0  |          0   |       30 



# Testing RNN (test.npz)

In [ ]:
folds:list[Path]=[item for item in Path(CV_PATH).iterdir() if item.is_dir()]
MODEL_PATHS: list[Path]=[Path(fold)/f"/model.keras" for fold in folds]
models:list[MyModel]=[keras.models.load_model(p) for p in MODEL_PATHS]

X_win_test, y_win_test, pid_win_test=load_split(architecture="rnn",name="test",CACHE=CACHE)

probs_win=np.mean([model.predict(X_win_test, batch_size=256, verbose=0) for model in models], axis=0)

yt, yp, _=patientaggregator.agg.aggregate(
    probs=probs_win,
    y_win=y_win_test,
    pids=pid_win_test
)

weighted_accuracy(y_true=yt, y_pred=yp)

In [ ]:
#Print full confusion matrix table 
md_table_build(csv_file=PATIENT_CM_FULL_PATH)

In [ ]:
#Print mean and standard deviation weighted accuracy table
md_table_build(csv_file=MEAN_WA_PATH)

In [ ]:
#Print additional metrics table: fold_f1-score.csv
md_table_build(csv_file=METRICS_CV_PATH)

## Alterações na estratégia de treino e previsão

**Suavização dos pesos das classes no treino (Smoothed Class Weights) com a raiz quadrada**: Substituir os pesos agressivos pela raiz quadrada da frequência inversa das classes (normalizada para média $1.0$), permitindo que a rede calibre probabilidades mais realistas sem ficar demasiado polarizada, mantendo a ponderação 5:3:1 apenas na avaliação final.

**Agregação dos pacientes: Top-K = 5** : em vez de calcularmos a média de todas as janelas temporais de cada paciente, fazemos a média das 5 janelas do doente que têm probabilidade mais elevada de sopro (Present). Evita que um sopro nítido em poucas gravações/janelas seja diluído por várias janelas sem sinal audível.

**Decisão `unweighted` em vez de `weighted`**: o argmax passa a ser aplicado diretamente às probabilidades do doente, sem as multiplicar por [5, 3, 1]. O WA continua a ser calculado com os pesos 5:3:1 na avaliação.


**Estas alterações encontram-se no ficheiro train_strategy.py e podem ser passadas como argumento na chamada do modelo CNN2D.**




## Resultados obtidos com as alterações:

- treino com 1 split:

                    WA_murmur: 0.8296

                                        True Present  True Unknown  True Absent
                            Pred Present            27             5           18
                            Pred Unknown             0             1            1
                            Pred Absent              0             4           86

- treino com k-fold cross-validation, k=5:

                        fold	    seed   n_doentes	     WA_fold	 epochs	    melhor_val_wa_interno	
                        0	     42	       159	     0.7741	     22	            0.7826	
                        1	     42	       160	     0.7533	     26	            0.7323	
                        2	     42	       160	     0.7484	     16	            0.6818	
                        3	     42	       160	     0.7336	     17	            0.8384	
                        4	     42	       161	     0.8479	     21	            0.8232	

                        media			0.7715			
                        dp			0.0451			

#### **O k-fold cross-validation está definido no ficheiro kfold_cv.py**


![curvas k-fold](figs/kfold_slide.png)

## **Teste:**

### Utilizamos um ensemble dos 5 modelos treinados na validação cruzada

Para a avaliação final usámos o conjunto de teste (`test.npz`), que nunca foi usado no treino, na validação interna (early stopping) nem na escolha de hiperparâmetros. Os modelos avaliados são os 5 obtidos na validação cruzada estratificada por doente (5 folds, `StratifiedGroupKFold`), todos treinados com a mesma configuração (pesos de classe `[smoothed]`, seed 42).

**Procedimento:**

1. Cada um dos 5 modelos (um por fold) prevê as probabilidades de cada janela do conjunto de teste.
2. As probabilidades são **médias entre os 5 modelos**, janela a janela (ensemble por média de probabilidades).
3. As previsões das janelas são agregadas por doente com **Top-K** (K = 5), e a classe final é o argmax das probabilidades agregadas.
4. Calculamos o WA_murmur (pesos 5:3:1 para Present:Unknown:Absent) e a matriz de confusão ao nível do doente.



**Resultados CNN2D**
- Pesos suavizados;
- Agregação Top K=5;
- Decisão não ponderada (unweighted).


                            WA_murmur (Teste - Ensemble 5 Folds): 0.7852

                                              True Present  True Unknown  True Absent
                            Pred Present            22             3            6
                            Pred Unknown             0             1            0
                            Pred Absent              5             6           99

